# B3 Stage-1 — combined sign DETECTOR (GTSDB + RTSD)

Отдельный ноутбук только для детектора. Проблема: немецкий YOLO **слеп к русским
синим квадратам** (recall 0.03-0.09 на RTSD). Решение — обучить 1-классовый
детектор `sign` на немецких (GTSDB) **и** русских (RTSD) сценах сразу:
чинит синие квадраты, сохраняет немецкое качество.

## Один раз ДО запуска
Залей `C:\mitgo\mitgo_det.zip` на Google Drive (`MyDrive/`) **под именем
`mitgo_det.zip`**. Если такое имя уже есть на Drive — сначала удали старое
(Drive не перезаписывает, создаёт `(1)`).

Внутри: `gtsdb_yolo/`, `rtsd_yolo/`, `mitgo_det.yaml` (проверяется ниже).

Runtime → Change runtime type → **GPU (T4)**, затем ячейки сверху вниз.

In [ ]:
!nvidia-smi -L
import torch; print('torch', torch.__version__, 'CUDA', torch.cuda.is_available())

In [ ]:
%cd /content
![ -d MITGOTSD ] || git clone https://github.com/damir095/MITGOTSD.git
%cd /content/MITGOTSD
!git pull --ff-only || true
!git log --oneline -1

In [ ]:
# У Colab уже CUDA-torch — НЕ переустанавливать torch.
!pip -q install ultralytics

In [ ]:
# Mount Drive + ПРЕДПРОВЕРКА архива ДО распаковки.
from google.colab import drive
drive.mount('/content/drive')

DET_ZIP  = '/content/drive/MyDrive/mitgo_det.zip'   # <-- имя на Drive
DET_ROOT = '/content/det_data'

import zipfile, os
assert os.path.exists(DET_ZIP), f'НЕТ файла на Drive: {DET_ZIP}'
_z  = zipfile.ZipFile(DET_ZIP)
_nm = [x.replace(chr(92), '/') for x in _z.namelist()]
_has = lambda p: any(p in x for x in _nm)
print('размер архива :', f'{os.path.getsize(DET_ZIP)/1e6:.0f} MB')
print('entries       :', len(_nm))
print('mitgo_det.yaml:', any(x.endswith('mitgo_det.yaml') for x in _nm))
print('gtsdb train   :', _has('gtsdb_yolo/images/train/'))
print('rtsd train    :', _has('rtsd_yolo/images/train/'))
print('rtsd val      :', _has('rtsd_yolo/images/val/'))
assert all([any(x.endswith('mitgo_det.yaml') for x in _nm),
            _has('gtsdb_yolo/images/train/'),
            _has('rtsd_yolo/images/train/')]), 'НЕ ТОТ архив — перезалей mitgo_det.zip'
print('\nOK — архив правильный.')

In [ ]:
# Чистая распаковка + правка path в yaml под Colab + проверка кадров.
import shutil, pathlib, glob
shutil.rmtree(DET_ROOT, ignore_errors=True)
pathlib.Path(DET_ROOT).mkdir(parents=True, exist_ok=True)
_z.extractall(DET_ROOT)

YAML = glob.glob(f'{DET_ROOT}/**/mitgo_det.yaml', recursive=True)[0]
lines = pathlib.Path(YAML).read_text().splitlines()
root = str(pathlib.Path(YAML).parent)
lines = [f'path: {root}' if l.startswith('path:') else l for l in lines]
pathlib.Path(YAML).write_text(chr(10).join(lines) + chr(10))
print(pathlib.Path(YAML).read_text())

for d in ('gtsdb_yolo', 'rtsd_yolo'):
    for s in ('train', 'val'):
        n = len(glob.glob(f'{root}/{d}/images/{s}/*'))
        print(f'{d}/{s}: {n} images')

## Обучение
Сначала smoke (2 эпохи) — убедиться, что данные/yaml читаются, потом полный прогон.
Старт с `yolov8n.pt`: combined-набор уже содержит GTSDB, поэтому немецкое не
теряется, а с нуля по знаку — чище, чем дообучать только-немецкий чекпойнт.

In [ ]:
!cd /content/MITGOTSD && yolo detect train model=yolov8n.pt data={YAML} \
    imgsz=640 batch=16 epochs=2 device=0 project=experiments/yolo name=smoke

In [ ]:
# Полный прогон. T4 16 GB. ~2-3 часа на ~9k+ кадров.
!cd /content/MITGOTSD && yolo detect train model=yolov8n.pt data={YAML} \
    imgsz=640 batch=32 epochs=100 patience=20 device=0 \
    project=experiments/yolo name=combined

In [ ]:
# Сохранить combined-детектор на Drive + скачать.
import shutil, glob
bp = sorted(glob.glob('/content/MITGOTSD/**/combined*/weights/best.pt',
                       recursive=True))[-1]
shutil.copy(bp, '/content/drive/MyDrive/yolo_combined.pt')
print('на Drive: yolo_combined.pt  (из', bp, ')')
from google.colab import files; files.download(bp)

## Назад на локальную машину
Скачанный **`yolo_combined.pt`** положи в `project/` — дальше я разложу в
`project/runs/detect/experiments/yolo/gtsdb/weights/best.pt` (бэкап старого),
и проверю обе стороны:
* `python -m src.evaluate_b3` — немецкое не просело,
* `python -m src.evaluate_yolo_rtsd` — синие квадраты 43/44/45 теперь видны.